# Notebook 3 - Surface water: did rivers and wet areas expand?

<a target="_blank" href="https://colab.research.google.com/github/khouakhi/UMP_EO_training/blob/main/notebooks/03_surface_water_change.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>


## Why this matters

Heavy rain can widen **river channels**, refill **reservoirs**, and expand **wetlands** or **temporary ponds**. Optical [**Sentinel-2 SR Harmonized**](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED) composites are easy to interpret but **clouds** block the view. [**Sentinel-1 GRD**](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S1_GRD) radar sees through clouds and is sensitive to **rough water** and **wet soil** (interpret with care).

## Research questions

1. Where was **open water / very wet surfaces** in the **wet season ending April 2025** versus **ending April 2026**?
2. Which areas look **new or expanded** in **2025/26** compared with **2024/25**?

## Outputs

- **MNDWI** median composites from [**Sentinel-2 SR Harmonized**](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED) (wet seasons **2024/25** vs **2025/26**), on the map in a separate cell.
- A simple **mask** of "more water-like in 2026" using a fixed MNDWI threshold (demonstration level, not an operational flood map).
- Optional [**Sentinel-1 GRD**](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S1_GRD) **VH** wet-season medians and a **blue** demo layer for **lower VH in 2026** (radar sees through clouds; interpret with care).

**Time tip:** about **35–40 minutes**.


In [ ]:
# Install packages (Colab often needs a fresh install each session)
# !pip install -q earthengine-api geemap

import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

# If you run this notebook locally, run `ee.Authenticate()` once before `ee.Initialize`.
# If the Colab pop-up fails, try: ee.Authenticate(auth_mode="colab")
# Mapping notebooks use `Map.add_basemap("SATELLITE")` so you always have photo context under EE layers.


In [ ]:
# Connect to Google Earth Engine using your cloud project ID.
# Set this to your own Google Earth Engine cloud project ID before running.
EE_PROJECT = "YOUR_GEE_PROJECT_ID"
EE_PROJECT = "ee-gee-hydro"

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("Earth Engine initialised with project:", EE_PROJECT)


In [ ]:
# Study area: Moulouya basin - HydroSHEDS level-8 hydrological unit (WWF)
# Dataset: WWF/HydroSHEDS/v1/Basins/hybas_8 - use the same HYBAS_ID in every notebook for consistency.

HYBAS_ID = 1080030220

MOULOUYA_BASIN_H08 = ee.FeatureCollection("WWF/HydroSHEDS/v1/Basins/hybas_8").filter(
    ee.Filter.eq("HYBAS_ID", HYBAS_ID)
)

# Geometry used for clips, filterBounds, reduceRegion, etc.
LOWER_MOULOUYA_AOI = MOULOUYA_BASIN_H08.geometry()

# Extended winter–spring wet season (December–April), named by the April that closes the window.


def wet_season_filter_dates(april_year: int) -> tuple[str, str]:
    # Returns filterDate(start, end) with end exclusive; April is fully included.
    start = f"{april_year - 1}-12-01"
    end = f"{april_year}-05-01"
    return (start, end)


## Sentinel-2: cloud-screened wet-season median

We use [**Harmonised Sentinel-2 SR**](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED) (`COPERNICUS/S2_SR_HARMONIZED`). MNDWI combines **green** and **SWIR** bands (roughly: bright water → higher values). We take a **median** across all **December–April** images in each wet season to reduce cloud noise.

**Scale factor:** reflectance values are divided by **10_000** in the harmonised collection. The **next cell** builds the two median images; the **cell after that** opens the map.


In [ ]:
SCALE = 1 / 10_000


def add_mndwi(img: ee.Image) -> ee.Image:
    g = img.select("B3").multiply(SCALE)
    s = img.select("B11").multiply(SCALE)
    mndwi = g.subtract(s).divide(g.add(s)).rename("MNDWI")
    return img.addBands(mndwi)


def s2_wet_season_mndwi_median(april_year: int) -> ee.Image:
    start, end = wet_season_filter_dates(april_year)
    s2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(LOWER_MOULOUYA_AOI)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 70))
        .map(add_mndwi)
    )
    return s2.median().clip(LOWER_MOULOUYA_AOI).select("MNDWI")


mndwi_apr2025 = s2_wet_season_mndwi_median(2025)
mndwi_apr2026 = s2_wet_season_mndwi_median(2026)


In [ ]:
visw = {"min": -0.5, "max": 0.6, "palette": ["c9940c", "fff7bc", "74add1", "023858"]}
aoi_vis = {"color": "red"}

Map = geemap.Map()
Map.add_basemap("SATELLITE")
Map.centerObject(LOWER_MOULOUYA_AOI, 9)
Map.addLayer(mndwi_apr2025, visw, "MNDWI Apr25")
Map.addLayer(mndwi_apr2026, visw, "MNDWI Apr26")
Map.addLayer(LOWER_MOULOUYA_AOI, aoi_vis, "AOI", opacity=0.2)
Map


## Simple “more water in 2026” mask

We subtract the two MNDWI images from [**Sentinel-2**](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED). Pixels with a **positive difference** above a small cutoff are highlighted. This is a **teaching threshold**, not calibrated for legal or emergency use.


In [ ]:
d_mndwi = mndwi_apr2026.subtract(mndwi_apr2025).rename("dMNDWI")
water_gain = d_mndwi.gt(0.12)
d_mndwi_vis = {"min": -0.3, "max": 0.3, "palette": ["b35806", "f7f7f7", "542788"]}
water_gain_vis = {"palette": ["0033ff"]}
aoi_vis = {"color": "red"}

Map2 = geemap.Map()
Map2.add_basemap("SATELLITE")
Map2.centerObject(LOWER_MOULOUYA_AOI, 9)
Map2.addLayer(d_mndwi, d_mndwi_vis, "dMNDWI")
Map2.addLayer(water_gain.updateMask(water_gain), water_gain_vis, "H2O+ demo")
Map2.addLayer(LOWER_MOULOUYA_AOI, aoi_vis, "AOI", opacity=0.2)
Map2


## Optional - Sentinel-1 VH (cloud-free cross-check)

[**Sentinel-1 GRD**](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S1_GRD) **VH** is often **lower** on smooth **open water** than on rough land (very simplified). **Wet soil** and **irrigation** also change VH, so treat this as a **qualitative** check next to MNDWI, not a second copy of the same story.

The **next cell** builds wet-season **VH medians** and a **blue** mask where VH **drops** from **2024/25** to **2025/26** (demo threshold, same idea as the MNDWI "more water-like" mask). The **cell after** draws the map.


In [ ]:
def s1_vh_wet_season_median(april_year: int) -> ee.Image:
    start, end = wet_season_filter_dates(april_year)
    s1 = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(LOWER_MOULOUYA_AOI)
        .filterDate(start, end)
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .select("VH")
    )
    return s1.median().clip(LOWER_MOULOUYA_AOI).rename("VH")


vh2025 = s1_vh_wet_season_median(2025)
vh2026 = s1_vh_wet_season_median(2026)
d_vh = vh2026.subtract(vh2025).rename("dVH")
# Demo: stronger VH decrease in 2026 flags as "more water-like" in blue (tune dB threshold for your AOI).
s1_water_demo = d_vh.lt(-1.5)


In [ ]:
vh_vis = {"min": -25, "max": -5, "palette": ["#0d0d0d", "#bdbdbd", "#ffffff"]}
d_vh_vis = {"min": -4, "max": 4, "palette": ["2166ac", "f7f7f7", "b2182b"]}
s1_water_vis = {"palette": ["0066ff"]}
aoi_vis = {"color": "red"}

Map4 = geemap.Map()
Map4.add_basemap("SATELLITE")
Map4.centerObject(LOWER_MOULOUYA_AOI, 9)
Map4.addLayer(vh2025, vh_vis, "VH Apr25")
Map4.addLayer(vh2026, vh_vis, "VH Apr26")
Map4.addLayer(d_vh, d_vh_vis, "dVH", opacity=0.55)
Map4.addLayer(s1_water_demo.updateMask(s1_water_demo), s1_water_vis, "S1 H2O+")
Map4.addLayer(LOWER_MOULOUYA_AOI, aoi_vis, "AOI", opacity=0.2)
Map4


## Discuss

1. Do **river corridors** and the **coastal lowlands** show the clearest **MNDWI increase**?
2. Name **one limitation** of using a **fixed MNDWI threshold** across crops, soil, and urban areas.
3. Does the optional **S1** blue mask line up with **MNDWI** water hints, or diverge (and why might radar disagree with green/SWIR optics)?

**Next:** `04_vegetation_ndvi_recovery.ipynb`.
